# PS17 — Loan Approval Prediction using Decision Tree and Naïve Bayes

**Aim:** Predict whether a loan is **Approved (1)** or **Rejected (0)** using **Decision Tree and Naïve Bayes**, compare accuracy, precision, recall, F1-score and confusion matrices.

**Dataset:** `loan_approval.csv`, a *synthetic educational dataset* included with the ZIP. It is not genuine bank data or a suitable model for actual lending decisions.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

from pathlib import Path

def find_csv(filename):
    choices = [
        Path.cwd() / filename,              # if notebooks + CSVs are together
        Path.cwd() / 'data' / filename,     # if CSVs are in data/
        Path.cwd().parent / filename,
        Path.cwd().parent / 'data' / filename,  # if notebook is inside notebooks/
    ]
    for candidate in choices:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f'{filename} not found. Upload the supplied CSV with the notebook '
        'or put it in a data/ folder. Checked: ' + str([str(x) for x in choices])
    )


## 2. Load and inspect the same CSV for all PS17–PS22

In [ ]:
df = pd.read_csv(find_csv('loan_approval.csv'))
print('Rows and columns:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head())
print('Data types:\n', df.dtypes)
print('Missing values:\n', df.isna().sum())
print('Class counts (0=Rejected, 1=Approved):\n', df['approved'].value_counts().sort_index())

## 3. Prepare X (features) and y (target)

Do not include `approved` in inputs. Split into **80% training / 20% testing** with stratification before fitting data transformers.

In [ ]:
required_cols = [
    'income', 'credit_score', 'employment_status',
    'loan_amount', 'loan_term', 'previous_default', 'approved'
]
assert set(required_cols).issubset(df.columns), 'CSV must contain all required columns'

df = df[required_cols].copy()
for col in ['income','credit_score','loan_amount','loan_term','previous_default','approved']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
# Missing target labels cannot be imputed; drop them.
df = df.dropna(subset=['approved']).copy()
df['approved'] = df['approved'].astype(int)
assert set(df['approved'].unique()).issubset({0,1})

X = df.drop(columns='approved')
y = df['approved']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 4. Preprocess data safely (no data leakage)

Numeric columns: median imputation + standardization. Categorical `employment_status`: mode imputation + one-hot encoding. `previous_default` is binary 0/1 and requires no encoding.

**Important:** the pipeline fits imputation, encoding and scaling on training data **only**, then transforms the test data.

In [ ]:
numeric_features = ['income', 'credit_score', 'loan_amount', 'loan_term']
categorical_features = ['employment_status']
binary_features = ['previous_default']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
    ('binary', binary_pipeline, binary_features)
])
print('Preprocessing pipeline is ready.')

## 5. Create and train Decision Tree and Naïve Bayes models

Both models use the same train/test split for a fair comparison.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Naive Bayes': GaussianNB(),
}

trained_models = {}
for name, classifier in models.items():
    model = Pipeline([
        ('preprocessing', clone(preprocessor)),
        ('classifier', classifier)
    ])
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(name, 'trained successfully.')

## 6. Evaluate: confusion matrix, accuracy, precision, recall, F1-score

**Confusion matrix:** rows = actual class, columns = predicted class. **Precision:** correct approvals among predicted approvals. **Recall:** correct approvals among actual approvals. **F1:** harmonic mean of precision and recall.

In [ ]:
results = []
for name, model in trained_models.items():
    predictions = model.predict(X_test)
    acc = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    results.append({
        'Algorithm': name,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1
    })
    print('\n', '='*45, '\n', name)
    cm = confusion_matrix(y_test, predictions, labels=[0,1])
    display(pd.DataFrame(cm,
        index=['Actual Rejected (0)', 'Actual Approved (1)'],
        columns=['Predicted Rejected (0)', 'Predicted Approved (1)']))
    print(classification_report(y_test, predictions, labels=[0,1],
          target_names=['Rejected (0)', 'Approved (1)'], zero_division=0))

results_df = pd.DataFrame(results)
print('Model comparison:')
display(results_df.round(3))

## 7. Compare models using a bar chart and report the higher F1

In [ ]:
ax = results_df.set_index('Algorithm')[
    ['Accuracy','Precision','Recall','F1-score']
].plot(kind='bar', figsize=(8,4), ylim=(0,1), rot=0)
ax.set_title('Loan Approval Classification — Model Comparison')
ax.set_ylabel('Score (0 to 1)')
plt.tight_layout()
plt.show()

best = results_df.sort_values('F1-score', ascending=False).iloc[0]
print(f"Higher test F1-score: {best['Algorithm']} ({best['F1-score']:.3f}).")
print('This ranking is specific to this synthetic dataset and split.')

## 8. Conclusion and viva points

- **Algorithms compared:** Decision Tree and Naïve Bayes.
- **Target variable:** `approved`; **1 = Approved**, **0 = Rejected**.
- **Train/test split:** 80:20; fixed random seed 42; stratified by target.
- **Why encode employment?** Text categories cannot be passed directly to these sklearn classifiers; one-hot avoids inventing an order.
- **Why standardize?** SVM and KNN are particularly affected by feature magnitude; keeping shared preprocessing makes comparisons straightforward.
- **Accuracy:** overall correct predictions; **precision** and **recall** distinguish different kinds of errors; **F1-score** summarizes their balance.
- **Confusion matrix:** TN, FP, FN, TP; rows show actual labels, columns predictions.
- **Caution:** No synthetic-data model is fit for real credit decisions; fairness, bias, calibration and legal review would be essential in reality.

**Practical exam tip:** To adapt to another problem statement, change only the two lines defining `models` in Step 5.